# Analyse des données ESS (Round 11)

**Source des données** : [European Social Survey (ESS), Round 11](https://ess.sikt.no/en/series/50ebb530-c72e-4002-a7bf-5581e3fdfe21) — fichier `ESS11MD`.

**Étape 1 — Lecture et chargement du fichier**

On ne conserve que :
- les variables **socio-démographiques** (pays, genre, âge, éducation, situation familiale, revenu, habitat, activité) ;
- la **confiance interpersonnelle** (`ppltrst`, `pplfair`, `pplhlp`) ;
- la **confiance envers les institutions** (`trstprl`, `trstlgl`, `trstplc`, `trstplt`, `trstprt`, `trstep`, `trstun`) ;
- les **valeurs humaines de Schwartz** — 21 items au format PVQ (dans ce fichier, suffixe `a` : `ipcrtiva`, `impricha`, …). *N.B. : l'ESS mesure les valeurs de Schwartz, pas la liste de valeurs (LOV) de Kahle* ;
- les **mesures de bonheur / bien-être** (`happy`, `stflife`, `health`) ;
- les **pondérations** (`anweight`, `pspwght`, `dweight`).

Le fichier CSV fait ~1,6 Go : on ne lit que les colonnes utiles (`usecols`).

In [1]:
import numpy as np
import pandas as pd

DATA_PATH = "data/ESS11MD_e01_2.csv"

pd.set_option("display.max_columns", 60)

In [2]:
# Lecture de l'en-tête seulement pour inventorier les colonnes disponibles
header = pd.read_csv(DATA_PATH, nrows=0)
all_cols = list(header.columns)
print(f"Nombre de colonnes dans le fichier : {len(all_cols)}")
print(all_cols[:20], "...")

Nombre de colonnes dans le fichier : 9348
['idno', 'cntry', 'ess11_reg', 'regunit', 'nuts1', 'nuts2', 'nuts3', 'name', 'essround', 'edition', 'proddate', 'dweight', 'pspwght', 'pweight', 'anweight', 'nwspol', 'netusoft', 'netustm', 'ppltrst', 'pplfair'] ...


## Sélection des variables

In [3]:
variables = {
    "identification": ["idno", "cntry"],
    "ponderations": ["anweight", "pspwght", "dweight"],
    "socio_demo": [
        "gndr",      # genre
        "agea",      # âge
        "eisced",    # niveau d'éducation (ES-ISCED)
        "eduyrs",    # années d'études
        "maritalb",  # statut marital
        "chldhhe",   # enfants au foyer (actuel/passé)
        "hhmmb",     # taille du ménage
        "domicil",   # type d'habitat (urbain/rural)
        "mnactic",   # activité principale
        "hinctnta",  # revenu du ménage (déciles)
        "hincfel",   # ressenti sur le revenu
    ],
    "confiance_interpersonnelle": ["ppltrst", "pplfair", "pplhlp"],
    "confiance_institutions": [
        "trstprl",  # parlement national
        "trstlgl",  # système judiciaire
        "trstplc",  # police
        "trstplt",  # hommes/femmes politiques
        "trstprt",  # partis politiques
        "trstep",   # Parlement européen
        "trstun",   # Nations unies
    ],
    "valeurs_schwartz": [
        "ipcrtiva", "impricha", "ipeqopta", "ipshabta", "impsafea",
        "impdiffa", "ipfrulea", "ipudrsta", "ipmodsta", "ipgdtima",
        "impfreea", "iphlppla", "ipsucesa", "ipstrgva", "ipadvnta",
        "ipbhprpa", "iprspota", "iplylfra", "impenva", "imptrada",
        "impfuna",
    ],
    "bonheur": [
        "happy",    # bonheur (0-10)
        "stflife",  # satisfaction dans la vie (0-10)
        "health",   # santé subjective (1-5)
    ],
}

keep_cols = [v for group in variables.values() for v in group]

manquantes = [v for v in keep_cols if v not in all_cols]
print(f"Variables sélectionnées : {len(keep_cols)}")
print(f"Variables absentes du fichier : {manquantes if manquantes else 'aucune'}")

Variables sélectionnées : 50
Variables absentes du fichier : aucune


## Chargement du fichier (colonnes sélectionnées uniquement)

⏳ Le fichier est volumineux, la lecture peut prendre une à deux minutes.

In [4]:
df = pd.read_csv(DATA_PATH, usecols=keep_cols)
df = df[keep_cols]  # réordonner selon nos groupes
print(f"Dimensions : {df.shape[0]:,} répondants x {df.shape[1]} variables")
df.head()

Dimensions : 46,162 répondants x 50 variables


,idno,cntry,anweight,pspwght,dweight,gndr,agea,eisced,eduyrs,maritalb,chldhhe,hhmmb,domicil,mnactic,hinctnta,hincfel,ppltrst,pplfair,pplhlp,trstprl,trstlgl,trstplc,trstplt,trstprt,trstep,trstun,ipcrtiva,impricha,ipeqopta,ipshabta,impsafea,impdiffa,ipfrulea,ipudrsta,ipmodsta,ipgdtima,impfreea,iphlppla,ipsucesa,ipstrgva,ipadvnta,ipbhprpa,iprspota,iplylfra,impenva,imptrada,impfuna,happy,stflife,health
0,50014,AT,0.130013,0.392891,1.185115,1,65,3,12,1,1,2,3,6,6,1,5,5,5,6,9,10,5,5,5,5,3,5,2,2,2,4,2,2,2,2,2,2,3,2,5,2,2,2,2,3,3,8,8,3
1,50030,AT,0.107598,0.325153,0.609898,2,21,5,14,6,2,1,1,2,1,2,10,0,1,6,6,4,1,0,5,5,2,4,2,4,4,4,4,2,2,2,2,1,4,2,4,2,4,1,1,4,2,9,9,2
2,50057,AT,1.323666,4.000023,1.392330,2,53,6,16,1,6,3,3,1,5,1,6,9,8,7,5,8,4,4,7,5,1,4,1,3,2,4,3,1,3,3,2,1,3,2,4,3,3,1,1,3,2,9,10,1
3,50106,AT,0.058316,0.176228,0.556061,2,78,5,14,4,1,1,1,6,2,2,6,6,6,5,6,9,3,3,4,4,3,4,2,3,1,4,3,3,2,3,2,2,3,3,4,3,3,2,2,2,3,7,7,3
4,50145,AT,0.351080,1.060940,0.722795,1,64,3,12,1,1,2,4,6,77,2,6,3,8,6,8,8,5,5,6,8,3,4,2,2,2,4,5,2,3,3,2,2,2,2,4,2,2,2,2,2,2,9,9,2


## Recodage des valeurs manquantes

Dans l'ESS, les non-réponses sont codées numériquement (ex. 77 = refus, 88 = ne sait pas, 99 = pas de réponse pour les échelles 0-10 ; 66/77/88/99 pour les items de valeurs en 1-6, etc.). On les convertit en `NaN`.

In [5]:
missing_codes = {}

# Échelles 0-10 : confiance et bien-être
for v in (variables["confiance_interpersonnelle"]
          + variables["confiance_institutions"]
          + ["happy", "stflife"]):
    missing_codes[v] = [77, 88, 99]

# Items de valeurs (1-6)
for v in variables["valeurs_schwartz"]:
    missing_codes[v] = [66, 77, 88, 99]

# Socio-démo (codes spécifiques)
missing_codes.update({
    "gndr":     [9],
    "agea":     [999],
    "eisced":   [55, 77, 88, 99],
    "eduyrs":   [77, 88, 99],
    "maritalb": [66, 77, 88, 99],
    "chldhhe":  [6, 7, 8, 9],
    "hhmmb":    [77, 88, 99],
    "domicil":  [7, 8, 9],
    "mnactic":  [66, 77, 88, 99],
    "hinctnta": [77, 88, 99],
    "hincfel":  [7, 8, 9],
    "health":   [7, 8, 9],
})

for v, codes in missing_codes.items():
    df[v] = df[v].replace(codes, np.nan)

# Taux de valeurs manquantes par variable
na_rate = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print("Taux de NA (%) — 15 premières variables :")
print(na_rate.head(15))

Taux de NA (%) — 15 premières variables :
chldhhe     33.3
hinctnta    21.3
trstun       7.3
trstep       6.4
ipstrgva     2.3
ipfrulea     2.3
trstlgl      2.1
trstprl      2.0
iprspota     2.0
ipudrsta     1.9
ipbhprpa     1.9
ipsucesa     1.9
ipcrtiva     1.8
trstprt      1.8
eduyrs       1.7
dtype: float64


## Aperçu des données

In [6]:
print("Répondants par pays :")
print(df["cntry"].value_counts())

df.describe().round(2).T.head(45)

Répondants par pays :
cntry
IT    2865
GR    2757
DE    2420
AT    2354
BG    2239
HU    2118
IE    2017
ES    1844
FR    1771
NL    1695
GB    1684
ME    1609
BE    1594
FI    1563
HR    1563
RS    1563
PL    1442
SK    1442
CH    1384
PT    1373
LT    1365
NO    1337
LV    1252
SI    1248
SE    1230
IL     906
IS     842
CY     685
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
idno,46162.0,72573.38,23106.05,50002.00,59679.75,69316.50,79059.75,211884.00
anweight,46162.0,0.93,1.40,0.00,0.18,0.41,1.15,13.00
pspwght,46162.0,1.00,0.61,0.04,0.65,0.90,1.17,6.13
dweight,46162.0,1.00,0.35,0.09,0.90,1.00,1.03,4.04
gndr,46162.0,1.53,0.50,1.00,1.00,2.00,2.00,2.00
agea,45805.0,51.64,18.72,15.00,37.00,52.00,67.00,90.00
eisced,45801.0,4.13,1.80,1.00,3.00,4.00,6.00,7.00
eduyrs,45356.0,13.25,4.09,0.00,11.00,13.00,16.00,69.00
maritalb,45419.0,3.14,2.26,1.00,1.00,1.00,6.00,6.00
chldhhe,30804.0,1.48,0.50,1.00,1.00,1.00,2.00,2.00


## Sauvegarde du sous-ensemble

On enregistre le jeu de données réduit au format CSV (`ESS_start.csv`) pour les étapes suivantes (évite de relire le CSV de 1,6 Go).

In [7]:
df.to_csv("data/ESS_start.csv", index=False)
print(f"Sauvegardé : data/ESS_start.csv ({df.shape[0]:,} lignes x {df.shape[1]} colonnes)")

Sauvegardé : data/ESS_start.csv (46,162 lignes x 50 colonnes)
